# BlueSky Starter Pack Data Collection

Team name: A5

Team members:
- **Trang Kieu:** Data collection and code review
- **Terresa Tran:** Edited fuctions to ensure they can run
- **Wynne Tseng:** Report analysis and results
- **Mei Wu:** Did thematic categorization code
- **Vivien Wang:** Review and orgainzation of report

### Q1: Qualitative: Concept Definition, Context, and Rationale


#### Context and Motivation

Bluesky starter packs are curated lists of accounts and feeds (up to 150 people and up to 3 custom feeds) designed to help users discover communities and content when joining the platform. Because Bluesky is decentralized and does not rely on traditional algorithmic recommendation systems to guide content discovery, users must depend more on manual or curated tools to navigate the network. As a result,  starter packs play a critical role in shaping user discovery and social network formation.

According to "Bootstrapping Social Networks: Lessons from Bluesky Starter Packs", starter packs often circulate within communities, creating clusters or "social bubbles" where users promote and follow others within the same social groups. This suggests that starter packs may reinforce community structures and amplify visibility of certain accounts.

Understanding these patterns helps answer broader questions about influence, discovery, and community formation in decentralized social networks.

#### Concept Definition: Echo Chamber

An echo chamber refers to a social environment in which users are primarily exposed to information, accounts, and viewpoints that align with their existing perspectives, while alternative perspectives remain marginal or absent.

In the context of Bluesky starter packs, echo chambers may form when starter packs repeatedly recommend accounts that are highly similar in terms of social connections, interests, views, or topical focus. This can lead to clustered communities where information circulates within the same group.

Because starter packs function as curated recommendation systems, they tend to introduce users to specific communities rather than broader and more diverse networks, it may contribute to echo chambers that users could become embedded in relatively homogenous clusters, reinforcing existing perspectives over time. 

##### We are going to define echo chamber through:
Follower overlap percentage between accounts in the same pack


#### Hypothesis

There are shared patterns across Bluesky starter packs. Specifically:
Some accounts appear frequently across many starter packs, indicating higher visibility or influence within the network.
Certain feeds are repeatedly included, suggesting they play a central role in content discovery.
Starter packs reflect community clusters or "social bubbles," in which users promote accounts within their own communities.

#### Research Questions

This phase focuses on four primary questions:

RQ1: Which accounts appear most frequently across starter packs?  
RQ2: Which feeds appear most frequently across starter packs?  
RQ3: Do starter pack descriptions suggest themes (art, politics, tech, sports, etc.)?  
RQ4: Do certain creators create many starter packs?

Starter packs may be organized around shared interests or communities.

#### Assumptions and Potential Biases
- Starter packs represent intentional curation and community knowledge
- Frequency of inclusion approximates visibility/importance
- Starter packs may overrepresent certain communities and underrepresent others depending on which URIs we could collect

### Constraints:
The Bluesky API returns details for a starter pack only when given a specific URI, so our approach depends on first collecting a list of starter pack URIs, then fetching each pack individually.

Also, for phase 1 of this project, we are just going to analyze 1,000 randomnized starter packs to see if our hypothesis is correct first, as there is 300,000 starter packs to analyze.

In [376]:
#!pip install atproto --quiet

In [377]:
# Import Libraries
import json

from atproto import Client, models
from atproto import exceptions
from password import BSKY_USERNAME, BSKY_APP_PASSWORD
import pandas as pd
from typing import List, Dict



In [378]:
# Enter your Bluesky Username and password for authentication
# Note: You can also create a file name password in the same directory and then store your user name as BSKY_USERNAME and password as BSKY_APP_PASSWORD.
# This Jupyter Notebook will import password file and your BSKY_USERNAME and BSKY_APP_PASSWORD variables automately
USERNAME = BSKY_USERNAME
APP_PASSWORD = BSKY_APP_PASSWORD

# Authenticate steps:
client = Client()
client.login(USERNAME, APP_PASSWORD)

ProfileViewDetailed(did='did:plc:lmc4xbbyqqyui7m6ptolv3lb', handle='tkieu137.bsky.social', associated=ProfileAssociated(activity_subscription=ProfileAssociatedActivitySubscription(allow_subscriptions='followers', py_type='app.bsky.actor.defs#profileAssociatedActivitySubscription'), chat=None, feedgens=0, labeler=False, lists=0, starter_packs=0, py_type='app.bsky.actor.defs#profileAssociated'), avatar='https://cdn.bsky.app/img/avatar/plain/did:plc:lmc4xbbyqqyui7m6ptolv3lb/bafkreig5n2ooeo3ixz4yfygzcuablb4ovfm7fijbq7dorzbsfu2ezxrefy@jpeg', banner=None, created_at='2026-01-13T22:59:12.525Z', debug=None, description=None, display_name='', followers_count=6, follows_count=75, indexed_at='2026-01-13T22:59:52.725Z', joined_via_starter_pack=None, labels=[], pinned_post=None, posts_count=4, pronouns=None, status=None, verification=None, viewer=ViewerState(activity_subscription=None, blocked_by=False, blocking=None, blocking_by_list=None, followed_by=None, following=None, known_followers=KnownFol

In [ ]:
# Read the starter packs dataset provided by Martin as a list of SP uri to gather data about the accounts within starter packs
df = pd.read_json("../starterpacks_65939_Terresa.jsonl", lines=True)
df


/var/folders/fq/8n_l7j9x6h9gn6knnd4lk4080000gn/T/ipykernel_48003/398822702.py:2: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_json("starterpacks_65939_Terresa.jsonl", lines=True)


ValueError: Expected object or value

### RQ1: Which account appear most in BlueSky Starter Packs?

Bluesky starter packs are curated lists of recommended accounts meant to help new users quickly find communities and high-quality content upon joining. Starter packs are created to mitigate "cold start" and were responsible for up to 43% of daily follow operations at their peak. Because each pack is created by a different user and focuses on a different theme or interest group, the accounts that appear most frequently across many starter packs are likely to be:

- broadly influential,

- highly visible across communities,

- Central hubs in the network.

Identifying these frequently-included accounts helps us understand:

- What kinds of voices are most prominent on Bluesky,

- Which users cross community boundaries,

- Whether certain media outlets, journalists, organizations, or personalities act as “anchor nodes” in the platform’s social ecosystem.

In [ ]:
def get_accounts_info(account_list: List) -> List[Dict]:
    """
    Takes a list of account objects (each one is one entry from a starter pack)
    and extracts just the important identity information we care about.

    Parameters
    ----------
    account_list : list
        A list of accounts. Each account contains a 'subject' field with
        information about the actual user (like DID and handle).

    Returns
    -------
    list[dict]
        A list of simple dictionaries. Each dictionary has:
        - "Account DID": the unique ID for the account
        - "Account Handler": the user's handle (e.g. 'nytimes.com')
    """
    list_accounts = []
    account_dict = {}
    for account in account_list:
        # Each "account" is actually a dictionary-like structure
        # pulled from the starter pack. Inside it, "subject" stores
        # the actual user profile. We extract the two fields we care about.
        account_did = account["did"]    #old code: account["subject"]["did"]
        account_handler = account["handle"]    #old code: account["subject"]["handle"]
        account_dict = {"Account DID": account_did,
                        "Account Handler": account_handler}
        list_accounts.append(account_dict)
    return list_accounts

In [ ]:
def get_starter_pack_info(starter_pack) -> dict:
    starter_pack_dict = {"SP DID": starter_pack["starter_pack"]["cid"],                            # unique ID for this starter pack record
                        "SP Creator DID": starter_pack["starter_pack"]["creator"]["did"],          # DID of the user who created the pack
                        "SP Creator Handle": starter_pack["starter_pack"]["creator"]["handle"],    # their handle (username)
                        "SP Description": starter_pack["starter_pack"]["record"]["description"]}   # text description of the pack}
    return starter_pack_dict

In [ ]:
def get_feed_info(starting_feed_list: List) -> List[Dict]:
    feed_list = []
    for feed in starting_feed_list:
        feed_list.append({"Feed DID": feed["cid"],
                        "Feed CID": feed["did"],
                        "Feed Description": feed["description"],
                        "Feed Creator DID": feed["creator"]["did"],
                        "Feed Creator Handle": feed["creator"]["handle"],
                        "Feed Like Count": feed["like_count"]})
    return feed_list

In [ ]:
from typing import List, Dict, Any

def get_all_accounts(list_uri: str) -> List[Any]:
    """
    Fetch ALL accounts in a list (app.bsky.graph.list) and return a flat list
    of profile objects (subjects).

    Each returned element is either:
    - a dict with keys like "did", "handle", ...
    - or a ProfileView/ProfileViewDetailed object from atproto_client.
    """
    accounts: List[Any] = []
    cursor = None

    while True:
        params = {"list": list_uri, "limit": 100}
        if cursor:
            params["cursor"] = cursor

        # Call get_list API
        res = client.app.bsky.graph.get_list(params=params)

        # if res has attribute 'items' store the info, if not, return an empty list
        items = res.items

        if not items:
            break

        for item in items:
            # check if subject is an attribute in the item and returns subject (account info),
            # if not, return items and append info
            subject = getattr(item, 'subject', item)
            accounts.append(subject)

        cursor = res.cursor

        if not cursor:
            break

    return accounts

In [ ]:
#@ return -> list[list[dict]]
def process_starter_pack_uri(uri: str) -> List[List[Dict]]:
    """
    Given a list of starter pack URIs, download information about each starter pack
    and the (sample of) accounts included in it.

    For each starter pack URI, we:
      1. Call the Bluesky API to get a detailed view of that starter pack.
      2. Extract metadata about the starter pack (who created it, description, etc.).
      3. Extract a sample list of accounts that appear in that starter pack.
      4. Flatten this into one row per (starter pack, account) pair.

    Parameters
    ----------
    uris : list
        A list of starter pack AT-URIs (strings). Each URI identifies one starter pack.

    Returns
    -------
    list[dict]
        A list of dictionaries. Each dictionary is one row linking:
        - a specific starter pack
        - one account that appears in that pack (from the sample list)
    """
    # this will hold all rows across all starter packs
    starter_pack_accounts_list = []
    feed_list = []

    sb_dict = {}

    proccessed_list = []

    try:
        # 1. Ask the Bluesky API for detailed information about this starter pack
        starter_pack = client.app.bsky.graph.get_starter_pack(params={"starterPack": uri})
    except exceptions.BadRequestError:
        # If the server says "starter pack not found", we skip this URI and continue.
        print("Skipping URI (starter pack not found):", uri)
        return {"starter_pack_rows": [], "feed_rows": []}
    except Exception as e:
        # catch-all to avoid killing the whole script
        print(f"Error fetching starter pack for {uri}: {e}")
        return {"starter_pack_rows": [], "feed_rows": []}

    # 2) Extract some basic metadata about this starter pack with get_starter_pack_info()
    starter_pack_info = get_starter_pack_info(starter_pack)

    # 3) Get the all accounts that appear in this starter pack
    # pass list of account uris from sp
    try:
        unproccessed_account_list_from_sb = get_all_accounts(starter_pack["starter_pack"]["list"]["uri"])
        # Use our helper to extract just DID + handle for each account in the sample
        proccessed_list = get_accounts_info(unproccessed_account_list_from_sb)

    except Exception as e:
        print(f"Error getting accounts for {uri}: {e}")
        processed_accounts = []

    try:
        if len(starter_pack["starter_pack"]["feeds"]) != 0:
            feed_info = get_feed_info(starter_pack["starter_pack"]["feeds"])
            for feed in feed_info:
                feed_list.append({"SP URI": uri,
                            "SP DID": starter_pack_info["SP DID"],
                            "SP Creator DID": starter_pack_info["SP Creator DID"],
                            "SP Creator Handle": starter_pack_info["SP Creator Handle"],
                            "SP Description": starter_pack_info["SP Description"],
                            "Feed DID": feed["Feed DID"],
                            "Feed CID": feed["Feed CID"],
                            "Feed Description": feed["Feed Description"],
                            "Feed Creator DID": feed["Feed Creator DID"],
                            "Feed Creator Handle": feed["Feed Creator Handle"],
                            "Feed Like Count": feed["Feed Like Count"]
                            })
    except Exception as e:
        print(f"Error getting feeds for {uri}: {e}")

    # 4) For each account in this starter pack's sample, create one flat row
    if proccessed_list:
      for account in proccessed_list:
          starter_pack_accounts_list.append({"SP URI": uri,
                          "SP DID": starter_pack_info["SP DID"],
                          "SP Creator DID": starter_pack_info["SP Creator DID"],
                          "SP Creator Handle": starter_pack_info["SP Creator Handle"],
                          "SP Description": starter_pack_info["SP Description"],
                          "Account DID": account["Account DID"],
                          "Account Handler": account["Account Handler"]})
      print(starter_pack_accounts_list)

    return {"starter_pack_rows": starter_pack_accounts_list,
            "feed_rows": feed_list
    }
    #return [starter_pack_accounts_list,feed_list]

In [ ]:
def process_uris_to_jsonl(uris: List[str], sp_accounts_jsonl_path: str, feeds_jsonl_path: str, processed_log_path = "processed_uris.txt"):
    # Keep track fo processed uri so we dont process one uri twice
    processed_uris = set()
    try:
        with open(processed_log_path, "r") as f:
            for line in f:
                processed_uris.add(line.strip())
    except FileNotFoundError:
        pass  # first run

    # 2) Open JSONL files in append mode
    with open(sp_accounts_jsonl_path, "a", encoding="utf-8") as sp_f, \
         open(feeds_jsonl_path, "a", encoding="utf-8") as feed_f, \
         open(processed_log_path, "a", encoding="utf-8") as log_f:

        for uri in uris:
            if uri in processed_uris:
                print(f"Skipping already-processed URI: {uri}")
                continue

            print(f"Processing URI: {uri}")
            result = process_starter_pack_uri(uri)

            # Write SP–account rows
            for row in result["starter_pack_rows"]:
                sp_f.write(json.dumps(row, ensure_ascii=False) + "\n")

            # Write feed rows
            for row in result["feed_rows"]:
                feed_f.write(json.dumps(row, ensure_ascii=False) + "\n")

            # Mark URI as processed (for resume)
            log_f.write(uri + "\n")
            log_f.flush()


### Creating Testing Dataset

In [ ]:
test_100 = df[:100]
test_1000 = df[:1000]

test_100.head()

,list,name,$type,createdAt,cid,author,uri,rkey,collection_time,feeds,updatedAt,description,descriptionFacets,image
0,at://did:plc:ccptzdrgiw457u2co7pyrnoy/app.bsky...,Kit de démarrage de 💖LOVHELENE💖,app.bsky.graph.starterpack,2025-10-02T18:22:08.416Z,bafyreidxptfvpt3ejvsa7jd2rthdrpmtw5y7tbnqniczy...,did:plc:ccptzdrgiw457u2co7pyrnoy,at://did:plc:ccptzdrgiw457u2co7pyrnoy/app.bsky...,3m2a6c4tqmz23,2026-02-05 22:14:08.109,[{'uri': 'at://did:plc:z72i7hdynmk6r22z27h6tvu...,None,Soutiens LFI et politiques honnêtes et photos ...,None,NaN
1,at://did:plc:ccptzdrgiw457u2co7pyrnoy/app.bsky...,Kit de démarrage de LOVHELENE,app.bsky.graph.starterpack,2025-01-28T13:48:58.572Z,bafyreieaedrra2a7pup2ujjfrpoquqzhlzpnrjaby7pjx...,did:plc:ccptzdrgiw457u2co7pyrnoy,at://did:plc:ccptzdrgiw457u2co7pyrnoy/app.bsky...,3lgslucnm3c27,2026-02-05 22:14:08.109,None,None,None,None,NaN
2,at://did:plc:vgcxikthzsg4cs5hdlpg7472/app.bsky...,‪damonsanders.bsky.social‬'s Starter Pack,app.bsky.graph.starterpack,2025-01-21T07:47:30.556Z,bafyreif6c5ol2rh3r4vrghekwm3b4iogj65eslb4hvt2b...,did:plc:vgcxikthzsg4cs5hdlpg7472,at://did:plc:vgcxikthzsg4cs5hdlpg7472/app.bsky...,3lgaefjbvww2n,2026-02-05 22:14:18.080,[{'uri': 'at://did:plc:tenurhgjptubkk5zf5qhi3o...,None,None,None,NaN
3,at://did:plc:7zixbdhgauk22zpwapbwnrqd/app.bsky...,Mga Mandirigma ng Liwanag!,app.bsky.graph.starterpack,2024-11-17T15:49:29.866Z,bafyreifosnnfdtlnkk7nkfn6yq5kirgsegwxbqxyb4ido...,did:plc:7zixbdhgauk22zpwapbwnrqd,at://did:plc:7zixbdhgauk22zpwapbwnrqd/app.bsky...,3lb5qzj453b2w,2026-02-05 22:14:24.242,[],2024-12-03T23:00:53.468Z,Sa akin! Mga mandirigma ng liwanag!!\nHere is ...,None,NaN
4,at://did:plc:na6akqmsdscdmdmnqeuflsmg/app.bsky...,Mandelbaum's Starter Pack,app.bsky.graph.starterpack,2025-01-07T02:45:27.056Z,bafyreiar4z5qz3t5u3iuviw6on33hsw3wkbeiz63b4u3s...,did:plc:na6akqmsdscdmdmnqeuflsmg,at://did:plc:na6akqmsdscdmdmnqeuflsmg/app.bsky...,3lf4mygzmvp23,2026-02-05 22:14:25.711,None,None,None,None,NaN


### Run the Script to Get Starter Packs Accounts and Starter Packs Feeds Dataset

In [ ]:
sp_output = "test_sp_accounts_tt65k.jsonl"
feed_output = "test_feeds_tt65k.jsonl"
log_output = "test_processed_tt65k.txt"

process_uris_to_jsonl(
    uris=df["uri"],
    sp_accounts_jsonl_path=sp_output,
    feeds_jsonl_path=feed_output,
    processed_log_path=log_output
)

Skipping already-processed URI: at://did:plc:ccptzdrgiw457u2co7pyrnoy/app.bsky.graph.starterpack/3m2a6c4tqmz23
Skipping already-processed URI: at://did:plc:ccptzdrgiw457u2co7pyrnoy/app.bsky.graph.starterpack/3lgslucnm3c27
Skipping already-processed URI: at://did:plc:vgcxikthzsg4cs5hdlpg7472/app.bsky.graph.starterpack/3lgaefjbvww2n
Skipping already-processed URI: at://did:plc:7zixbdhgauk22zpwapbwnrqd/app.bsky.graph.starterpack/3lb5qzj453b2w
Skipping already-processed URI: at://did:plc:na6akqmsdscdmdmnqeuflsmg/app.bsky.graph.starterpack/3lf4mygzmvp23
Skipping already-processed URI: at://did:plc:na6akqmsdscdmdmnqeuflsmg/app.bsky.graph.starterpack/3lf4n2rdlyd2o
Skipping already-processed URI: at://did:plc:ltuq4ot7wutyp3se2sjgqqkg/app.bsky.graph.starterpack/3ltjpnjkebr2f
Skipping already-processed URI: at://did:plc:2rtmzqsr5ucsqu6tarwznp5j/app.bsky.graph.starterpack/3law6hn6jkd2c
Skipping already-processed URI: at://did:plc:xtfssbavew65vgwmm5kzh7h3/app.bsky.graph.starterpack/3lb7sshmzpy2d
S

KeyboardInterrupt: 